In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

from functools import partial

import numpy as np
from astropy import units as u
from astropy.coordinates import ICRS
from astropy.table import QTable
from astropy_healpix import HEALPix
from ligo.skymap import plot  # noqa: F401
from ligo.skymap.util import progress_map_vectorized
from m4opt.fov import footprint_healpix
from m4opt.missions import uvex as mission
from m4opt.utils.numpy import count_intersect1d_combinations
from matplotlib import colors
from matplotlib import pyplot as plt
from regions import Regions
from tqdm.auto import tqdm

In [ ]:
(bounding_rectangle,) = Regions.read("../fov/bounding-rectangle.ds9")
(inscribed_circle,) = Regions.read("../fov/inscribed-circle.ds9")
plan = QTable.read("../tables/initial-survey.ecsv")

## Time usage

In [ ]:
total_time = u.Quantity(list(plan.meta["total_time"].values())).to(u.day)
action = list(plan.meta["total_time"].keys())
plt.pie(
    total_time,
    labels=action,
    autopct=lambda pct: (0.01 * pct * total_time.sum().to(u.day)).round(2),
)
plt.savefig("../visualizations/time-utilization.pdf", metadata={"CreationDate": None})

## Fraction of sky visited N times

In [ ]:
obs = plan[plan["action"] == "observe"]
hpx = HEALPix(nside=2048, frame=ICRS())

fovs = [bounding_rectangle, inscribed_circle, mission.fov]
fov_names = ["Bounding rectangle", "Inscribed circle", "Chips"]
visits = []
visit_maps = []

for fov in fovs:
    footprints = progress_map_vectorized(
        partial(footprint_healpix, hpx, fov),
        obs["target_coord"].unmasked,
        obs["roll"].unmasked,
        jobs=None,
    )
    visit_maps.append(
        visit_map := np.bincount(np.concatenate(footprints), minlength=hpx.npix)
    )
    visits.append(np.bincount(visit_map))
max_visits = max(len(v) for v in visits)
visits = np.stack([np.pad(v, (0, max_visits - len(v))) for v in visits])

In [ ]:
fig_width, fig_height = plt.rcParams["figure.figsize"]
nrows = len(fov_names)
fig, axs = plt.subplots(
    nrows,
    1,
    figsize=(fig_width, 0.6 * nrows * fig_height),
    dpi=300,
    subplot_kw={"projection": "astro aitoff", "center": "8h 0d"},
)
for ax, fov_name, visit_map in zip(axs, fov_names, visit_maps):
    im = ax.imshow_hpx(
        visit_map, norm=colors.LogNorm(vmin=0.8, vmax=max_visits, clip=True)
    )
    ax.set_title(fov_name)
    ax.grid()
    fig.colorbar(im, ax=ax).set_label("Number of visits")
plt.savefig("../visualizations/visit-map.pdf", metadata={"CreationDate": None})

In [ ]:
fig_width, fig_height = plt.rcParams["figure.figsize"]
nrows = len(fov_names)
bins = np.concatenate(
    [[0]]
    + [
        np.arange(1, 10) * 10**order
        for order in range(int(np.ceil(np.log10(max_visits))))
    ]
)
bins = bins[bins <= max_visits]
fig, axs = plt.subplots(
    nrows, 1, figsize=(fig_width, nrows * fig_height), sharex=True, sharey=True
)
for ax, fov_name, visit in zip(axs, fov_names, visits):
    ax.hist(np.arange(len(visit)), bins=bins, weights=visit / hpx.npix)
    ax.grid(which="both")
    ax.set_ylabel("Sky fraction")
    ax.set_title(fov_name)
ax.set_xlabel("Number of visits")
ax.set_ylim(0, None)
ax.set_xscale("log")
for ax in axs:
    twin = ax.twinx()
    twin.set_ylim(np.asarray(ax.get_ylim()) * 4 * 180**2 / np.pi / 1000)
    twin.set_ylabel("Sky area / $10^3$ deg$^2$")
plt.savefig("../visualizations/visit-distribution.pdf", metadata={"CreationDate": None})

## Cadence distribution

In [ ]:
start_mjd = obs["start_time"].mjd
weights = count_intersect1d_combinations(footprints)
i, j = np.triu_indices(len(start_mjd), 1)
delays = start_mjd[j] - start_mjd[i]

In [ ]:
v = np.bincount(np.bincount(np.concatenate(footprints), minlength=hpx.npix))
area_visited_at_most_once = (v[0] + v[1]) * hpx.pixel_area.to_value(u.deg**2)

for nbins in tqdm([2, 4, 8, 20, 40]):
    fig, axs = plt.subplots(1, 2, sharey=True, width_ratios=(20, 1))
    axs[0].hist(
        delays,
        weights=weights * hpx.pixel_area.to_value(u.deg**2),
        bins=np.logspace(
            np.floor(np.log10(delays.min())), np.ceil(np.log10(delays.max())), nbins + 1
        ),
    )
    axs[0].set_xscale("log")
    axs[0].set_xlabel("Time delay (days)")
    axs[0].set_ylabel("Area (deg$^2$)")
    axs[0].set_title("Cadence distribution")
    axs[0].grid()

    axs[1].bar("Visited\nat most once", area_visited_at_most_once)

    axs[0].spines.left.set_visible(False)
    axs[0].spines.top.set_visible(False)
    axs[0].spines.right.set_visible(False)
    axs[1].spines.left.set_visible(False)
    axs[1].spines.top.set_visible(False)
    axs[1].spines.right.set_visible(False)
    fig.savefig(
        f"../visualizations/cadence-distribution-{nbins}-bins.pdf",
        metadata={"CreationDate": None},
    )

## Slew distribution

In [ ]:
downlink_slew_angles = plan["slew_angle"][1:-1][
    ((plan[:-2]["action"] == "downlink") | (plan[2:]["action"] == "downlink"))
    & (plan[1:-1]["action"] == "slew")
].filled(np.nan)
non_downlink_slew_angles = plan["slew_angle"][1:-1][
    (plan[:-2]["action"] != "downlink")
    & (plan[2:]["action"] != "downlink")
    & (plan[1:-1]["action"] == "slew")
].filled(np.nan)

In [ ]:
fig_width, fig_height = plt.rcParams["figure.figsize"]
fig, axs = plt.subplots(
    2,
    1,
    figsize=(fig_width, 2 * fig_height),
    sharex=True,
    gridspec_kw={"hspace": 0.1},
)
axs[0].hist(
    non_downlink_slew_angles.to_value(u.deg), bins=np.arange(0, 181, 1), log=True
)
axs[0].set_ylabel("Observing slews only)")
axs[1].set_ylabel("Downlink slews only)")
axs[1].hist(downlink_slew_angles.to_value(u.deg), bins=np.arange(0, 181, 15), log=True)
axs[1].set_xlabel("Slew angle (deg)")
axs[1].set_xlim(0, 180)
axs[1].xaxis.set_major_locator(plt.MultipleLocator(15))
fig.suptitle("Slew angle frequency distribution")
fig.savefig(
    "../visualizations/slew-angle-distribution.pdf",
    metadata={"CreationDate": None},
)